In [2]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\acer\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\acer\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\acer\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [3]:
df = pd.read_csv('../data/raw/FR_Dataset.csv')

In [4]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def clean_text(text):
    # 1 lower the text
    text = text.lower()

    # 2 remove punctuation/special characters
    text = re.sub(r'[^a-zA-Z\s]','',text)

    # 3 split into words(tokenization)
    words = text.split()
    
    # 4. remove stopwords AND lemmatize each remaining word
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]

    # 5. join back into a single string
    return ' '.join(words)

In [5]:
sample = df['text_'].iloc[0]
print("Before:",sample)
print("After:",clean_text(sample))

Before: Love this!  Well made, sturdy, and very comfortable.  I love it!Very pretty
After: love well made sturdy comfortable love itvery pretty


In [6]:
df['clean_text'] = df['text_'].apply(clean_text)


In [7]:
df[['text_' ,'clean_text']].sample(5)

,text_,clean_text
19161,This deadbolt was very easy to install and the...,deadbolt easy install material good reason gav...
31720,"V. is like some weird, dark fantasy, with a lo...",v like weird dark fantasy lot twist turn blown...
21707,I bought this for my dog (Border Collie mix) a...,bought dog border collie mix love comfortable ...
18445,I got it because it has a magnetic end. Which ...,got magnetic end strong hold piece metal want ...
28240,I loved the story. It was a good read. I wou...,loved story good read would recommend read boo...


In [8]:
df.to_csv('../data/processed/cleaned_reviews.csv', index=False)

Feature 1: Review length

In [9]:
df['review_length'] = df['text_'].apply(len)

Feature 2: Sentiment score using TextBlob

In [10]:
pip install textblob

   ---------------------------------------- 0.0/625.0 kB ? eta -:--:--
   ---------------------------------------- 625.0/625.0 kB 10.9 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [12]:
from textblob import TextBlob
df['sentiment'] = df['text_'].apply(lambda x: TextBlob(x).sentiment.polarity)

Feature 3: Sentiment-rating mismatch

In [13]:
df['scaled_rating'] = -1 + (df['rating'] - 1) * 0.5

df['sentiment_mismatch'] = abs(df['scaled_rating'] - df['sentiment'])

In [14]:
df[['rating', 'sentiment', 'scaled_rating', 'sentiment_mismatch']].sample(5)

,rating,sentiment,scaled_rating,sentiment_mismatch
23260,5.0,0.377544,1.0,0.622456
18487,4.0,0.199242,0.5,0.300758
25598,5.0,0.312500,1.0,0.687500
7329,5.0,0.354167,1.0,0.645833
25051,1.0,0.150000,-1.0,1.150000


In [15]:
df['exclamation_count'] = df['text_'].apply(lambda x: x.count('!'))

In [16]:
superlatives = ['amazing', 'perfect', 'best', 'excellent', 'awesome', 'incredible', 'love', 'great']

def count_superlatives(text):
    words = text.split()
    return sum(1 for word in words if word in superlatives)

df['superlative_count'] = df['clean_text'].apply(count_superlatives)

In [17]:
df[['clean_text', 'exclamation_count', 'superlative_count']].sample(5)

,clean_text,exclamation_count,superlative_count
1133,hold stack degree weather tool included great ...,0,1
37293,exactly described wear xl order wear,0,0
36202,prefer unpainted wooden product bought year ol...,2,5
26451,totally loved book couldnt put first book ive ...,0,1
37153,like pant material nice soft keep long time,0,0
